# 01 — Estandarización y tabla analítica canónica

**Proyecto:** Vigía Cali — sistema auditable de vigilancia temporal de la criminalidad reportada para la planeación institucional.

**Autores:** completar manualmente antes de la entrega.

## Propósito

Estandarizar las fuentes ya filtradas por Cali, conservar la
trazabilidad fila–archivo y construir una tabla canónica para EDA.
La fuente **Hurto por Modalidades** se mantiene separada porque puede
solaparse con Hurto a Personas.


## 1. Configuración y precondiciones


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

DATAV3_ROOT = Path("/content/drive/MyDrive/datav3")
SIEDCO_INPUT = DATAV3_ROOT / "A1 - SIEDCO" / "datos_criminalidad_cali"
PROJECT_OUTPUT = DATAV3_ROOT / "project_diplodata_outputs" / "eda_01_05_v1"
LANDING = PROJECT_OUTPUT / "landing"
TRUSTED = PROJECT_OUTPUT / "trusted"
SURFACE = PROJECT_OUTPUT / "surface"
AUDIT = PROJECT_OUTPUT / "audit"
REPORTS = PROJECT_OUTPUT / "reportes"

for directory in (LANDING, TRUSTED, SURFACE, AUDIT, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)

PERIODO_INICIO = 2018
PERIODO_FIN = 2025

from IPython.display import display

SIEDCO_LANDING = LANDING / "siedco"
DANE_LANDING = LANDING / "dane"
SOURCE_TRUSTED = TRUSTED / "fuentes_siedco"
SOURCE_TRUSTED.mkdir(exist_ok=True)

required_audits = (AUDIT / "inventario_siedco.csv", AUDIT / "inventario_dane.csv")
for required_path in required_audits:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Falta {required_path}. Ejecute 00_descargas.ipynb antes de continuar."
        )

audit_siedco = pd.read_csv(AUDIT / "inventario_siedco.csv")
if len(audit_siedco) != 8 or not audit_siedco["estado"].eq("OK").all():
    raise RuntimeError("La auditoría SIEDCO no está completa o contiene fallos.")


## 2. Estandarización fuente por fuente

No se agregan departamentos ni municipios completos. Las entradas son
exclusivamente las salidas de Cali generadas por el notebook 00.


In [ ]:
def normalize_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def find_column(columns, candidates, required=True):
    normalized = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        if normalize_name(candidate) in normalized:
            return normalized[normalize_name(candidate)]
    if required:
        raise KeyError(f"Falta una columna entre {candidates}: {list(columns)}")
    return None


def classify_modality(value):
    label = normalize_name(value)
    if "persona" in label:
        return "HURTO A PERSONAS"
    if "residenc" in label:
        return "HURTO A RESIDENCIAS"
    if "comerc" in label:
        return "HURTO A COMERCIO"
    if "motocic" in label:
        return "HURTO DE MOTOCICLETAS"
    if "automotor" in label or "vehicul" in label:
        return "HURTO DE AUTOMOTORES"
    return "OTRA MODALIDAD DE HURTO"


canonical_parts = []
complement_parts = []
consolidation_log = []

for path in sorted(SIEDCO_LANDING.glob("*.csv")):
    frame = pd.read_csv(path, low_memory=False)
    required_columns = {
        "_fuente_id", "_archivo_origen", "_tipo_fuente", "_rol_fuente",
        "_fecha_estandar", "_cantidad_estandar", "_fila_origen",
    }
    missing = required_columns.difference(frame.columns)
    if missing:
        raise KeyError(f"{path.name}: faltan columnas de trazabilidad {missing}")

    standardized = pd.DataFrame(
        {
            "fuente_id": frame["_fuente_id"],
            "archivo_origen": frame["_archivo_origen"],
            "fila_origen": frame["_fila_origen"],
            "rol_fuente": frame["_rol_fuente"],
            "tipo_fuente": frame["_tipo_fuente"],
            "fecha": pd.to_datetime(frame["_fecha_estandar"], errors="coerce"),
            "cantidad": pd.to_numeric(frame["_cantidad_estandar"], errors="coerce"),
        }
    )
    standardized["anio"] = standardized["fecha"].dt.year.astype("Int64")
    standardized["mes"] = standardized["fecha"].dt.month.astype("Int64")
    dias_semana = {0: "lunes", 1: "martes", 2: "miércoles", 3: "jueves", 4: "viernes", 5: "sábado", 6: "domingo"}
    standardized["dia_semana"] = standardized["fecha"].dt.dayofweek.map(dias_semana)

    source_role = standardized["rol_fuente"].dropna().unique()
    if len(source_role) != 1:
        raise ValueError(f"{path.name}: rol de fuente inconsistente.")

    if source_role[0] == "principal":
        source_types = standardized["tipo_fuente"].dropna().unique()
        if len(source_types) != 1:
            raise ValueError(f"{path.name}: tipo de fuente inconsistente.")
        standardized["tipo_delito"] = source_types[0]
        canonical_parts.append(standardized)
        included_rows = len(standardized)
        excluded_overlap = 0
    else:
        descriptor = find_column(
            frame.columns,
            (
                "delito", "tipo_delito", "descripcion_conducta",
                "conducta", "descripcion", "modalidad",
            ),
        )
        standardized["valor_modalidad_origen"] = frame[descriptor].astype("string")
        standardized["tipo_delito"] = standardized[
            "valor_modalidad_origen"
        ].map(classify_modality)
        standardized["solapa_hurto_personas"] = standardized[
            "tipo_delito"
        ].eq("HURTO A PERSONAS")
        standardized["incluible_canonica"] = standardized["tipo_delito"].isin(
            {
                "HURTO A RESIDENCIAS",
                "HURTO A COMERCIO",
                "HURTO DE MOTOCICLETAS",
                "HURTO DE AUTOMOTORES",
            }
        )
        complement_parts.append(standardized)
        non_overlapping = standardized.loc[
            standardized["incluible_canonica"]
        ].copy()
        canonical_parts.append(non_overlapping)
        included_rows = len(non_overlapping)
        excluded_overlap = int(standardized["solapa_hurto_personas"].sum())

    standardized.to_csv(
        SOURCE_TRUSTED / path.name, index=False, encoding="utf-8-sig"
    )
    consolidation_log.append(
        {
            "archivo_landing": path.name,
            "rol_fuente": source_role[0],
            "filas_entrada": len(standardized),
            "filas_canonicas": included_rows,
            "filas_solapadas_hurto_personas_excluidas": excluded_overlap,
        }
    )


## 3. Decisión de solapamiento

Hurto por Modalidades no se apila completo:

- Las filas clasificadas como Hurto a Personas se excluyen de la tabla
  canónica porque esa conducta ya tiene una fuente principal.
- Residencias, comercio, motocicletas y automotores se incluyen solo
  cuando la categoría de origen permite identificarlas.
- Categorías desconocidas permanecen en la tabla complementaria y no
  afectan los totales canónicos.


In [ ]:
canonical = pd.concat(canonical_parts, ignore_index=True)
complement = pd.concat(complement_parts, ignore_index=True)
consolidation_log = pd.DataFrame(consolidation_log)

if canonical["cantidad"].isna().any() or (canonical["cantidad"] < 0).any():
    raise ValueError("La tabla canónica contiene cantidades inválidas.")
if canonical["fecha"].isna().any():
    print(
        "Advertencia: existen fechas inválidas. El notebook 02 las cuantificará "
        "y bloqueará los análisis temporales afectados."
    )
if complement.empty:
    raise ValueError("No se generó la fuente complementaria esperada.")

canonical.to_csv(
    TRUSTED / "analitica_canonica.csv", index=False, encoding="utf-8-sig"
)
complement.to_csv(
    TRUSTED / "hurto_modalidades_complemento.csv",
    index=False,
    encoding="utf-8-sig",
)
consolidation_log.to_csv(
    AUDIT / "consolidacion_fuentes.csv", index=False, encoding="utf-8-sig"
)

display(consolidation_log)
display(
    canonical.groupby("tipo_delito", as_index=False)["cantidad"]
    .sum()
    .sort_values("cantidad", ascending=False)
)


## 4. Comprobación e interpretación

La tabla anterior es una comprobación estructural, no un resultado EDA
validado. El número de filas no se interpreta como casos. Si faltan
categorías necesarias para las preguntas 1–5, la fase posterior debe
bloquearse y revisar la clasificación de la fuente complementaria.


In [ ]:
expected_crimes = {
    "HOMICIDIO", "HURTO A PERSONAS", "HURTO A RESIDENCIAS",
    "HURTO A COMERCIO", "HURTO DE MOTOCICLETAS",
    "HURTO DE AUTOMOTORES", "LESIONES PERSONALES",
    "VIOLENCIA INTRAFAMILIAR", "DELITOS SEXUALES",
    "AMENAZAS", "EXTORSION",
}
observed_crimes = set(canonical["tipo_delito"].dropna().unique())
missing_crimes = sorted(expected_crimes - observed_crimes)
check = pd.DataFrame(
    {
        "comprobacion": [
            "Fuentes auditadas", "Categorías canónicas esperadas",
            "Filas con cantidad inválida", "Fechas inválidas",
        ],
        "resultado": [
            len(consolidation_log),
            f"{len(observed_crimes & expected_crimes)}/{len(expected_crimes)}",
            int(canonical["cantidad"].isna().sum()),
            int(canonical["fecha"].isna().sum()),
        ],
    }
)
display(check)
if missing_crimes:
    raise RuntimeError(
        "No se puede continuar: faltan categorías EDA: " + ", ".join(missing_crimes)
    )
